# Linear Attention and Deep SSMs

**Prerequisites:** This chapter builds on [Transformers](04_04_transformers) and [Linear Dynamical Systems](04_02_lds). The state space model perspective connects linear attention to the structured SSMs developed there.

Softmax attention is the engine of the Transformer, but its $O(N^2)$ cost in sequence length limits its applicability to long sequences. A family of models — **linear attention**, **structured state space models (SSMs)**, and their selective and learnable-rule variants — achieve $O(N)$ computation by maintaining a fixed-size **hidden state** that is updated recurrently. This chapter develops the mathematics connecting these architectures, from the kernel view of [@katharopoulos2020transformers] through S4 [@gu2022efficiently], Mamba [@gu2023mamba], DeltaNet [@yang2024parallelizing], and test-time training [@yu2024learning].


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch import nn


## Standard Softmax Attention

Given queries $Q \in \mathbb{R}^{N \times d}$, keys $K \in \mathbb{R}^{N \times d}$, and values $V \in \mathbb{R}^{N \times d}$, the output of **causal** (autoregressive) softmax attention at position $i$ is:

$$
y_i = \frac{\sum_{j=1}^{i} \exp\!\left(q_i^\top k_j / \sqrt{d}\right) v_j}{\sum_{j=1}^{i} \exp\!\left(q_i^\top k_j / \sqrt{d}\right)}.
$$

In matrix form (with the causal mask $M_{ij} = \mathbb{I}[j \leq i]$):

$$
Y = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + \log M\right) V,
$$

where $\log M$ sets future entries to $-\infty$. Computing this requires materializing the $N \times N$ attention matrix, costing $O(N^2 d)$ time and $O(N^2)$ memory — prohibitive for long sequences.


## Linear Attention: The Kernel Trick

[@katharopoulos2020transformers] observe that the softmax in attention is a kernel function:

$$
\exp(q^\top k / \sqrt{d}) = \kappa(q, k).
$$

For a **general positive-definite kernel** $\kappa(q, k) = \phi(q)^\top \phi(k)$ with explicit feature maps $\phi: \mathbb{R}^d \to \mathbb{R}^r$, the causal attention output becomes:

$$
y_i = \frac{\sum_{j \leq i} \phi(q_i)^\top \phi(k_j)\, v_j}{\sum_{j \leq i} \phi(q_i)^\top \phi(k_j)}
= \frac{\phi(q_i)^\top \sum_{j \leq i} \phi(k_j) v_j^\top}{\phi(q_i)^\top \sum_{j \leq i} \phi(k_j)}.
$$

The crucial observation: $\phi(q_i)$ acts on the **accumulated outer products** $\sum_{j \leq i} \phi(k_j) v_j^\top \in \mathbb{R}^{r \times d}$, which can be built up **incrementally**. Defining the hidden state and normalizer:

$$
S_t = S_{t-1} + \phi(k_t) v_t^\top \in \mathbb{R}^{r \times d}, \qquad z_t = z_{t-1} + \phi(k_t) \in \mathbb{R}^r,
$$

the output at each step is:

$$
y_t = \frac{S_t^\top \phi(q_t)}{z_t^\top \phi(q_t)}.
$$

This is a **linear RNN** with hidden state $S_t$ — no quadratic cost, no materializing the attention matrix. The computation is $O(N r d)$ in time and $O(r d)$ in memory (for the hidden state).

A simple feature map that keeps values positive is $\phi(x) = \mathrm{elu}(x) + 1$ applied elementwise, so $r = d$.


In [ ]:
def feature_map(x):
    '''ELU+1 feature map; keeps activations non-negative.'''
    return F.elu(x) + 1


def linear_attention_parallel(Q, K, V):
    '''Causal linear attention in parallel (matrix) form. O(N^2 r d).

    Args:
        Q, K, V: (N, d) tensors
    Returns:
        Y: (N, d) output
    '''
    Qf = feature_map(Q)   # (N, r), r == d here
    Kf = feature_map(K)

    # Causal kernel matrix: A[i,j] = phi(q_i)^T phi(k_j), j <= i
    A = torch.tril(Qf @ Kf.T)                    # (N, N)
    Z = A.sum(dim=-1, keepdim=True).clamp(min=1e-6)
    return (A / Z) @ V


def linear_attention_recurrent(Q, K, V):
    '''Causal linear attention in recurrent form. O(N r d).

    Args:
        Q, K, V: (N, d) tensors
    Returns:
        Y: (N, d) output
    '''
    N, d = Q.shape
    Qf = feature_map(Q)
    Kf = feature_map(K)

    S = torch.zeros(d, d)   # hidden state: r x d  (r == d)
    z = torch.zeros(d)      # normalizer
    Y = torch.zeros(N, d)

    for t in range(N):
        S = S + torch.outer(Kf[t], V[t])    # rank-1 update
        z = z + Kf[t]
        denom = (Qf[t] @ z).clamp(min=1e-6)
        Y[t] = (Qf[t] @ S) / denom

    return Y


# Verify the two forms agree
torch.manual_seed(0)
N, d = 16, 8
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

Y_par = linear_attention_parallel(Q, K, V)
Y_rec = linear_attention_recurrent(Q, K, V)
print(f'Max difference between parallel and recurrent forms: {(Y_par - Y_rec).abs().max():.2e}')


## Structured State Space Models

### Continuous-Time SSMs

A **linear time-invariant (LTI) state space model** describes a continuous-time dynamical system:

$$
h'(t) = A\, h(t) + B\, x(t), \qquad y(t) = C^\top h(t),
$$

where $h(t) \in \mathbb{R}^p$ is the hidden state, $x(t) \in \mathbb{R}$ is the scalar input, and $A \in \mathbb{R}^{p \times p}$, $B \in \mathbb{R}^p$, $C \in \mathbb{R}^p$ are parameters.

Discretizing with step size $\Delta$ (zero-order hold) gives the **recurrent** form:

$$
h_t = \bar{A}\, h_{t-1} + \bar{B}\, x_t, \qquad y_t = C^\top h_t,
$$

where $\bar{A} = e^{\Delta A}$ and $\bar{B} = (\bar{A} - I) A^{-1} B$.

Because $\bar{A}$ is fixed, the output is a **convolution** of the input with the SSM kernel:

$$
y_t = \sum_{s=0}^{t} \underbrace{C^\top \bar{A}^{t-s} \bar{B}}_{\bar{K}_{t-s}} x_s = (\bar{K} * x)_t,
$$

where $\bar{K} = (C^\top \bar{B},\; C^\top \bar{A} \bar{B},\; C^\top \bar{A}^2 \bar{B}, \ldots)$ is the **SSM kernel** (impulse response). This **duality** — recurrent for inference, convolutional for training — is the key computational advantage of SSMs.

### S4 and the HiPPO Initialization

The $S4$ model [@gu2022efficiently] builds on the **HiPPO** framework [@gu2020hippo], which initializes $A$ as a structured matrix designed to optimally memorize the history of the input via polynomial projections. The HiPPO-LegS matrix has entries:

$$
A_{nk} =
\begin{cases}
-(2n+1)^{1/2}(2k+1)^{1/2} & n > k \\
-(n+1) & n = k \\
0 & n < k
\end{cases}
$$

Computing $e^{\Delta A}$ naively is $O(p^3)$. S4 exploits the fact that HiPPO-LegS is a **normal plus low-rank** (NPLR) matrix to compute the SSM kernel in $O(p + N \log N)$ time via the fast Fourier transform.

### S5: Diagonal SSMs with Parallel Scans

$S5$ [@smith2023simplified] simplifies S4 by diagonalizing the state matrix: $\Lambda = P^{-1} A P$ where $\Lambda$ is diagonal. With $\tilde{h}_t = P^{-1} h_t$, the recurrence decouples into $p$ independent scalar recurrences:

$$
\tilde{h}_{t,i} = \lambda_i \tilde{h}_{t-1,i} + \tilde{B}_i x_t, \qquad y_t = \mathrm{Re}(C^\top \tilde{h}_t).
$$

Each mode decays independently at rate $|\lambda_i|$; stability requires $|\lambda_i| < 1$. S5 computes all $N$ hidden states in parallel using an **associative parallel scan**, reducing training time from $O(Np)$ sequential steps to $O(p \log N)$ parallel steps.


In [ ]:
def ssm_recurrent(x, A_bar, B_bar, C):
    '''Scalar-input LTI SSM in recurrent form.

    Args:
        x:     (N,) input sequence
        A_bar: (p, p) discrete state matrix
        B_bar: (p,) discrete input vector
        C:     (p,) output vector
    Returns:
        y: (N,) output sequence
    '''
    N = len(x)
    p = len(C)
    h = torch.zeros(p)
    y = torch.zeros(N)
    for t in range(N):
        h = A_bar @ h + B_bar * x[t]
        y[t] = C @ h
    return y


def ssm_convolutional(x, A_bar, B_bar, C):
    '''Scalar-input LTI SSM in convolutional form (for comparison).

    Args:
        x:     (N,) input sequence
        A_bar: (p, p) discrete state matrix
        B_bar: (p,) discrete input vector
        C:     (p,) output vector
    Returns:
        y: (N,) output sequence via direct convolution with kernel K
    '''
    N = len(x)
    # Build SSM kernel K[t] = C^T A_bar^t B_bar
    K = torch.zeros(N)
    Apow = torch.eye(len(C))
    for t in range(N):
        K[t] = C @ (Apow @ B_bar)
        Apow = A_bar @ Apow
    # Causal convolution via FFT
    K_pad = torch.cat([K, torch.zeros(N)])
    x_pad = torch.cat([x, torch.zeros(N)])
    y_full = torch.fft.irfft(torch.fft.rfft(K_pad) * torch.fft.rfft(x_pad))
    return y_full[:N]


torch.manual_seed(1)
N, p = 64, 4

# Simple stable diagonal A_bar (eigenvalues in (0, 1))
lam = torch.tensor([0.9, 0.8, 0.7, 0.6])
A_bar = torch.diag(lam)
B_bar = torch.randn(p)
C = torch.randn(p)
x = torch.randn(N)

y_rec  = ssm_recurrent(x, A_bar, B_bar, C)
y_conv = ssm_convolutional(x, A_bar, B_bar, C)
print(f'Max diff recurrent vs convolutional: {(y_rec - y_conv).abs().max():.2e}')

# Visualise the SSM kernel and a response
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
K_vals = [float(C @ (torch.linalg.matrix_power(A_bar, t) @ B_bar)) for t in range(N)]
axes[0].stem(K_vals[:32], markerfmt='C0o', linefmt='C0-', basefmt='k-')
axes[0].set_title('SSM impulse response $\\bar{K}_t$')
axes[0].set_xlabel('Lag $t$')

axes[1].plot(x.numpy(), label='input $x_t$', alpha=0.6)
axes[1].plot(y_rec.numpy(), label='output $y_t$', lw=2)
axes[1].set_title('SSM input/output')
axes[1].set_xlabel('Time step')
axes[1].legend()

plt.tight_layout()
plt.savefig('ssm_demo.png', dpi=100, bbox_inches='tight')
plt.show()


## Selective State Space Models: Mamba

A fundamental limitation of LTI SSMs is that the transition matrices $\bar{A}$ and $\bar{B}$ are **input-independent**: every token is processed identically regardless of content. This prevents the model from selectively retaining or discarding information based on context.

**Mamba** [@gu2023mamba] introduces a **selection mechanism** by making $B_t$, $C_t$, and the discretization step $\Delta_t$ functions of the input $x_t$:

$$
h_t = \bar{A}_t\, h_{t-1} + \bar{B}_t\, x_t, \qquad y_t = C_t^\top h_t,
$$

where $\bar{A}_t = e^{\Delta_t A}$, $\bar{B}_t = (\bar{A}_t - I) A^{-1} B_t$, and $\Delta_t, B_t, C_t$ are computed from $x_t$ via small linear projections.

Because parameters now vary with $t$, the convolutional view no longer applies — the system is **time-varying**, and $y_t$ depends on all of $x_1, \ldots, x_t$ in a non-linear way. Training requires a **selective scan**: a hardware-aware parallel algorithm that exploits the structure of the recurrence without materializing intermediate states in memory.

The selection mechanism gives Mamba attention-like content-dependent routing while preserving the $O(N p)$ cost of a recurrent model.


## DeltaNet: The Delta Rule as Attention

### Online Least Squares and the Delta Rule

Consider maintaining a **weight matrix** $S_t \in \mathbb{R}^{d \times d}$ that approximates a key–value associative memory: given key $k_t$ we want $S_t k_t \approx v_t$. A natural objective is the online least-squares loss:

$$
\ell_t(S) = \tfrac{1}{2}\|S k_t - v_t\|^2.
$$

Taking one step of **gradient descent** with step size $\beta_t \in (0, 1]$ gives the **delta rule** update:

$$
S_t = S_{t-1} - \beta_t \nabla_S \ell_t(S_{t-1})
= S_{t-1} - \beta_t (S_{t-1} k_t - v_t) k_t^\top.
$$

Rearranging:

$$
\boxed{S_t = (I - \beta_t k_t k_t^\top)\, S_{t-1} + \beta_t v_t k_t^\top.}
$$

This is a **rank-1 correction**: it subtracts the current memory's prediction $S_{t-1} k_t$ (scaled by $\beta_t$) and adds the target $v_t$, weighted by the same key $k_t$.

Compare to linear attention:

$$
S_t^{\text{lin}} = S_{t-1} + \phi(k_t) v_t^\top. \qquad \text{(no forgetting)}
$$

DeltaNet replaces the identity transition with $(I - \beta_t k_t k_t^\top)$: old memories **associated with $k_t$** are selectively overwritten, not merely accumulated.

### DeltaNet as an Attention Model

**DeltaNet** [@yang2024parallelizing] uses the delta-rule hidden state and produces output by querying with $q_t$:

$$
y_t = S_t^\top q_t.
$$

With keys and queries $\ell_2$-normalized ($\|k_t\| = \|q_t\| = 1$) and $\beta_t \in (0, 1]$ a learned gate (e.g., via sigmoid), DeltaNet is a **gated linear RNN** with a content-dependent forgetting mechanism. It is more expressive than linear attention (which has $A = I$) but cheaper than softmax attention ($O(N d^2)$ vs. $O(N^2 d)$).

Training DeltaNet efficiently requires computing the delta-rule updates **in parallel** over the sequence. [@yang2024parallelizing] derive a chunk-wise algorithm based on the **WY representation** of Householder products that achieves this without sacrificing hardware efficiency.


In [ ]:
def deltanet_recurrent(Q, K, V, beta):
    '''DeltaNet recurrence: delta-rule hidden state queried by Q.

    Args:
        Q, K: (N, d) unit-normalized query / key tensors
        V:    (N, d) value tensors
        beta: (N,) gate values in (0, 1]
    Returns:
        Y:    (N, d) output
    '''
    N, d = Q.shape
    S = torch.zeros(d, d)
    Y = torch.zeros(N, d)
    for t in range(N):
        k, v, b = K[t], V[t], beta[t]
        # Delta rule: remove old association, write new one
        S = S - b * torch.outer(S @ k, k) + b * torch.outer(v, k)
        Y[t] = S.T @ Q[t]
    return Y


def linear_attention_recurrent_nodenom(Q, K, V):
    '''Linear attention without denominator normalisation (for direct comparison).'''
    N, d = Q.shape
    Qf = feature_map(Q)
    Kf = feature_map(K)
    S = torch.zeros(d, d)
    Y = torch.zeros(N, d)
    for t in range(N):
        S = S + torch.outer(Kf[t], V[t])
        Y[t] = Qf[t] @ S
    return Y


torch.manual_seed(7)
N, d = 32, 16
Q = F.normalize(torch.randn(N, d), dim=-1)
K = F.normalize(torch.randn(N, d), dim=-1)
V = torch.randn(N, d) * 0.5
beta = torch.sigmoid(torch.randn(N))   # learned gate

Y_delta = deltanet_recurrent(Q, K, V, beta)

# Hidden-state norms: DeltaNet should stay bounded, linear attention grows
Y_lin_nd = linear_attention_recurrent_nodenom(Q, K, V)

# Track norm of S over time for both
S_delta = torch.zeros(d, d)
S_lin   = torch.zeros(d, d)
norms_delta, norms_lin = [], []
Qf = feature_map(Q); Kf = feature_map(K)
for t in range(N):
    k, v, b = K[t], V[t], beta[t]
    S_delta = S_delta - b * torch.outer(S_delta @ k, k) + b * torch.outer(v, k)
    S_lin   = S_lin + torch.outer(Kf[t], V[t])
    norms_delta.append(S_delta.norm().item())
    norms_lin.append(S_lin.norm().item())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(norms_delta, label='DeltaNet $\\|S_t\\|_F$', color='tab:blue')
ax.plot(norms_lin,   label='Linear attn $\\|S_t\\|_F$', color='tab:orange', ls='--')
ax.set_xlabel('Step $t$')
ax.set_ylabel('Frobenius norm of $S_t$')
ax.set_title('Hidden state norm: DeltaNet vs. linear attention')
ax.legend()
plt.tight_layout()
plt.savefig('deltanet_demo.png', dpi=100, bbox_inches='tight')
plt.show()


## Test-Time Training and Test-Time Regression

### The Hidden State as a Learned Model

**Test-time training (TTT)** [@yu2024learning] takes the associative-memory perspective to its logical conclusion: the hidden state $W_t$ **is** the weights of a small model (e.g., a linear map or a two-layer MLP) that is updated **online** by gradient descent during the forward pass.

For a **linear model** with weights $W_t \in \mathbb{R}^{d \times d}$ and a self-supervised reconstruction loss:

$$
\ell_t(W) = \tfrac{1}{2}\|W k_t - v_t\|^2,
$$

one gradient step with learning rate $\eta$ gives:

$$
W_t = W_{t-1} - \eta (W_{t-1} k_t - v_t) k_t^\top.
$$

**This is exactly the delta rule** with $\beta_t = \eta$. TTT with a linear model and DeltaNet with a fixed gate $\beta$ are the same algorithm, viewed through different lenses: DeltaNet is motivated by online learning of an associative memory, while TTT is motivated by gradient-based adaptation of a hidden model.

The output in TTT is produced by applying the current model to the query:

$$
y_t = W_t q_t.
$$

For richer hidden models (two-layer MLP with parameters $W_t$), TTT uses mini-batches of context tokens and takes multiple gradient steps, creating a **meta-learned** inner loop. This trades off compute for expressivity of the hidden state.

### Test-Time Regression: A Probabilistic View

The connection to regression is made explicit by interpreting the context tokens $(k_1, v_1), \ldots, (k_t, v_t)$ as a **training set** and $q_t$ as a **test point**. The TTT linear model solves the online ridge regression problem:

$$
W_t = \arg\min_W \sum_{s \leq t} \|W k_s - v_s\|^2 + \lambda \|W\|_F^2.
$$

The closed-form solution is:

$$
W_t = \left(\sum_{s \leq t} v_s k_s^\top\right) \left(\sum_{s \leq t} k_s k_s^\top + \lambda I\right)^{-1},
$$

and the prediction $y_t = W_t q_t$ is **kernel ridge regression** evaluated at $q_t$ with a linear kernel. From this perspective:

- **Linear attention** (without denominator) computes $S_t q_t$ where $S_t = \sum_{s \leq t} v_s k_s^\top$ — this is the **numerator** of the ridge regression estimator, ignoring the Gram matrix $(\sum_s k_s k_s^\top + \lambda I)^{-1}$.
- **Standard attention** can be interpreted as kernel regression with the softmax kernel.
- **DeltaNet / TTT** tracks an online approximation to the full ridge regression solution, forgetting old training points in proportion to their alignment with the current key.

This regression perspective unifies the family of linear-complexity sequence models as variants of **in-context regression** with different choices of kernel, loss, and optimization algorithm.


## Summary

The table below organises the models in this chapter by their hidden-state update rule. All achieve $O(N)$ inference complexity; they differ in what information they can selectively retain.

| **Model** | **Hidden state update** | **Key property** |
|---|---|---|
| Linear attention [@katharopoulos2020transformers] | $S_t = S_{t-1} + \phi(k_t) v_t^\top$ | No forgetting; $A = I$ |
| S4 / S5 [@gu2022efficiently] [@smith2023simplified] | $h_t = \bar{A}\, h_{t-1} + \bar{B}\, x_t$ | Fixed structured $\bar{A}$; HiPPO init |
| Mamba [@gu2023mamba] | $h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t$ | Input-dependent $\bar{A}_t, \bar{B}_t$ |
| DeltaNet [@yang2024parallelizing] | $S_t = (I - \beta_t k_t k_t^\top) S_{t-1} + \beta_t v_t k_t^\top$ | Online gradient descent; selective overwrite |
| TTT (linear) [@yu2024learning] | $W_t = W_{t-1} - \eta (W_{t-1}k_t - v_t)k_t^\top$ | Gradient descent on reconstruction loss |

Two broad axes organise the design space:

1. **Selectivity**: can the model gate what it remembers based on content? ($A = I$ → no; Mamba, DeltaNet → yes).
2. **Expressivity of the hidden state**: scalar recurrence (S4) → matrix recurrence (linear attention, DeltaNet) → MLP weights (TTT).

The dominant computational pattern — a linear recurrence that admits parallel training via associative scans or convolutions — appears across all these architectures, connecting classical signal processing, online learning, and modern sequence modeling.
